# Exploration des Données – Challenge AutoML

Ce notebook permet d'explorer et de visualiser les **6 jeux de données** du challenge situés dans `/info/corpus/ChallengeMachineLearning` :
- `data_A`
- `data_B`
- `data_C`
- `data_D`
- `data_E`
- `data_F`

### Objectifs de l'exploration :
1. **Comprendre la structure** des fichiers (`.data`, `.type`, `.solution`).
2. **Identifier le type de tâche** pour chaque jeu de données (classification binaire, multiclasse, multi-label, régression).
3. **Détecter les spécificités techniques** (matrices denses vs creuses, types de variables, valeurs manquantes).
4. **Définir la stratégie minimale** pour notre pipeline AutoML.

In [5]:
import os
import glob
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

CORPUS_DIR = "/info/corpus/ChallengeMachineLearning"
DATASETS = ["data_A", "data_B", "data_C", "data_D", "data_E", "data_F"]
print(f"Jeux de données ciblés : {DATASETS}")

Jeux de données ciblés : ['data_A', 'data_B', 'data_C', 'data_D', 'data_E', 'data_F']


## 1. Fonction utilitaire de chargement (support Dense et Creux sans label)

In [6]:
def load_features(filepath, n_features=None):
    """Charge un fichier .data (dense ou creux index:valeur sans étiquette)."""
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
        
    if ':' in first_line:
        # Format creux LibSVM sans colonne de label au début
        rows, cols, data = [], [], []
        with open(filepath, 'r') as f:
            for r, line in enumerate(f):
                for item in line.strip().split():
                    c, v = item.split(':')
                    rows.append(r)
                    cols.append(int(c))
                    data.append(float(v))
        n_samples = r + 1
        min_c, max_c = min(cols), max(cols)
        if min_c == 1:
            cols = [c - 1 for c in cols]
            n_cols = n_features if n_features else max_c
        else:
            n_cols = n_features if n_features else max_c + 1
        return sp.csr_matrix((data, (rows, cols)), shape=(n_samples, n_cols))
    else:
        return np.loadtxt(filepath)

def inspect_dataset(name):
    folder = os.path.join(CORPUS_DIR, name)
    data_path = os.path.join(folder, f"{name}.data")
    type_path = os.path.join(folder, f"{name}.type")
    sol_path = os.path.join(folder, f"{name}.solution")
    
    # Types des colonnes
    col_types = []
    if os.path.exists(type_path):
        with open(type_path, 'r') as f:
            col_types = [l.strip() for l in f if l.strip()]
            
    # Cibles / solution
    sol = np.loadtxt(sol_path)
    
    # Descripteurs X
    X = load_features(data_path, n_features=len(col_types) if col_types else None)
    is_sparse = sp.issparse(X)
    shape = X.shape
    
    # Détection de la tâche
    if sol.ndim == 2 and sol.shape[1] > 1:
        is_one_hot = np.all(np.sum(sol, axis=1) == 1) and np.array_equal(sol, sol.astype(bool))
        if is_one_hot:
            task = f"Multiclasse One-Hot ({sol.shape[1]} classes)"
        else:
            task = f"Multi-Label ({sol.shape[1]} cibles binaires)"
    else:
        u_vals = len(np.unique(sol))
        if u_vals == 2:
            task = "Classification Binaire"
        elif u_vals < 20 and np.all(np.equal(np.mod(sol, 1), 0)):
            task = f"Classification Multiclasse ({u_vals} classes)"
        else:
            task = "Régression continue"
            
    return {
        "Dataset": name,
        "Lignes (N)": shape[0],
        "Features (D)": shape[1],
        "Format X": "Creux (Sparse CSR)" if is_sparse else "Dense",
        "Types Features": dict(pd.Series(col_types).value_counts()) if col_types else "Non spécifié",
        "Forme Solution": sol.shape,
        "Tâche Déduite": task,
        "sol_sample": sol[:5]
    }

## 2. Tableau comparatif des 6 jeux de données

In [7]:
results = []
detailed_info = {}
for name in DATASETS:
    info = inspect_dataset(name)
    detailed_info[name] = info
    results.append({
        "Dataset": info["Dataset"],
        "Lignes (N)": info["Lignes (N)"],
        "Colonnes (D)": info["Features (D)"],
        "Format": info["Format X"],
        "Forme Solution": str(info["Forme Solution"]),
        "Tâche Déduite": info["Tâche Déduite"],
        "Types Colonnes": str(info["Types Features"])
    })

df_summary = pd.DataFrame(results)
df_summary

,Dataset,Lignes (N),Colonnes (D),Format,Forme Solution,Tâche Déduite,Types Colonnes
0,data_A,34190,24,Dense,"(34190, 3)",Multi-Label (3 cibles binaires),"{'Categorical': np.int64(13), 'Numerical': np...."
1,data_B,5000,16,Dense,"(5000,)",Régression continue,{'Numerical': np.int64(16)}
2,data_C,15000,1568,Dense,"(15000, 10)",Multiclasse One-Hot (10 classes),{'Numerical': np.int64(1568)}
3,data_D,2984,144,Dense,"(2984,)",Classification Binaire,"{'Binary': np.int64(136), 'Numerical': np.int6..."
4,data_E,3140,259,Dense,"(3140,)",Classification Binaire,{'Numerical': np.int64(259)}
5,data_F,13142,61188,Creux (Sparse CSR),"(13142, 20)",Multiclasse One-Hot (20 classes),{'Numerical': np.int64(61188)}


## 3. Zoom sur chaque jeu de données

In [8]:
for name in DATASETS:
    info = detailed_info[name]
    print(f"==================== {name} ====================")
    print(f"Tâche : {info['Tâche Déduite']}")
    print(f"Format X : {info['Format X']} | Shape : ({info['Lignes (N)']}, {info['Features (D)']})")
    print(f"Types : {info['Types Features']}")
    print(f"Aperçu Solution (5 premières lignes) :\n{info['sol_sample']}")
    print()

==================== data_A ====================
Tâche : Multi-Label (3 cibles binaires)
Format X : Dense | Shape : (34190, 24)
Types : {'Categorical': np.int64(13), 'Numerical': np.int64(11)}
Aperçu Solution (5 premières lignes) :
[[1. 1. 1.]
 [1. 1. 0.]
 [0. 1. 1.]
 [1. 1. 0.]
 [1. 0. 1.]]

==================== data_B ====================
Tâche : Régression continue
Format X : Dense | Shape : (5000, 16)
Types : {'Numerical': np.int64(16)}
Aperçu Solution (5 premières lignes) :
[ 93600. 500000.  87900. 234600.  57600.]

==================== data_C ====================
Tâche : Multiclasse One-Hot (10 classes)
Format X : Dense | Shape : (15000, 1568)
Types : {'Numerical': np.int64(1568)}
Aperçu Solution (5 premières lignes) :
[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]

==================== data_D ====================
Tâche : Classification Binaire
Format X : Dense |